In [ ]:
import pandas as pd
import os

march_path = "/green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03"
april_path = "/green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-04"

night_records = []

def process_folder(folder_path):

    global night_records

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.gz'):
                full_path = os.path.join(root, file)
                print(f"Processing: {full_path}")

                try:
                    df = pd.read_csv(full_path, compression='gzip', sep='\t', header=None)
                    df.columns = ['id', 'lat', 'lon', 'ts', 'date', 'grid', 'type']

                    # STOP filter
                    df = df[df['type'] == 'stop']

                    # TIME
                    df['ts'] = pd.to_datetime(df['ts'], unit='s')
                    df['hour'] = df['ts'].dt.hour

                    # HOURLY DEDUP
                    df['hour_bin'] = df['ts'].dt.floor('h')
                    df = df.drop_duplicates(subset=['id', 'hour_bin'])

                    # NIGHT FILTER
                    df = df[(df['hour'] >= 19) | (df['hour'] <= 7)]

                    # NIGHT DATE FIX (important)
                    df['night_date'] = df['ts'].dt.date
                    df.loc[df['hour'] <= 7, 'night_date'] = df['ts'].dt.date - pd.Timedelta(days=1)

                    # WEEK
                    df['week'] = df['ts'].dt.isocalendar().week

                    # GROUP per night
                    night_stats = df.groupby(['id', 'night_date']).agg(
                        num_points=('ts', 'count'),
                        start_time=('ts', 'min'),
                        end_time=('ts', 'max')
                    ).reset_index()

                    # DURATION
                    night_stats['duration_hours'] = (
                        night_stats['end_time'] - night_stats['start_time']
                    ).dt.total_seconds() / 3600

                    # APPLY NIGHT FILTER CONDITIONS
                    night_stats = night_stats[
                        (night_stats['num_points'] >= 2) &
                        (night_stats['duration_hours'] >= 5)
                    ]

                    # ADD WEEK INFO AGAIN
                    night_stats['week'] = pd.to_datetime(night_stats['night_date']).dt.isocalendar().week

                    night_records.append(night_stats)

                    del df, night_stats

                except Exception as e:
                    print(f"Error processing {file}: {e}")

# RUN
process_folder(march_path)
process_folder(april_path)

# Combine
night_df = pd.concat(night_records, ignore_index=True)

print("Night-level records:", len(night_df))

Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00000.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00001.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00002.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00003.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00004.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00005.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00006.gz
Processing: /green-projects/project-urban_colocation_intelligence/workspace/share/data/2025-03/3/part-00007.gz
